In [7]:
import pandas as pd
import numpy as np
from astroquery.simbad import Simbad

# 1. Load the Nagata CSV you just downloaded
df = pd.read_csv("Nagata_1990_Lprime_polarization_standards.csv")

# 2. Use the star column directly for SIMBAD queries
# (strip whitespace to help matching)
df["SIMBAD_name"] = df["star"].astype(str).str.strip()

unique_names = df["SIMBAD_name"].unique()

# 3. Configure SIMBAD to return spectral type and photometry
s = Simbad()

# RA/DEC are included by default; we add spectral type + fluxes
s.add_votable_fields(
    "sp",                 # spectral type -> column SP_TYPE
    "flux(U)", "flux(B)", "flux(V)", "flux(R)", "flux(I)",
    "flux(J)", "flux(H)", "flux(K)"
)

records = []

for name in unique_names:
    try:
        res = s.query_object(name)
    except Exception as e:
        print(f"SIMBAD query failed for {name}: {e}")
        res = None

    if res is None or len(res) == 0:
        # No SIMBAD match
        records.append({
            "SIMBAD_name": name,
            "ra": "-",
            "dec": "-",
            "sp_type": "-",
            "U": "-", "B": "-", "V": "-", "R": "-", "I": "-",
            "J": "-", "H": "-", "K": "-"
        })
        continue

    row = res[0]

    def get(field):
        # robust getter that deals with masked values, NaNs, etc.
        try:
            val = row[field]
        except KeyError:
            return "-"
        # Masked values
        if hasattr(val, "mask") and getattr(val, "mask", False):
            return "-"
        # Regular None/NaN
        if val is None:
            return "-"
        if isinstance(val, float) and np.isnan(val):
            return "-"
        return val

    records.append({
        "SIMBAD_name": name,
        "ra": get("RA"),
        "dec": get("DEC"),
        "sp_type": get("SP_TYPE"),
        "U": get("FLUX_U"),
        "B": get("FLUX_B"),
        "V": get("FLUX_V"),
        "R": get("FLUX_R"),
        "I": get("FLUX_I"),
        "J": get("FLUX_J"),
        "H": get("FLUX_H"),
        "K": get("FLUX_K"),
    })

simbad_df = pd.DataFrame(records)

# 4. Merge back into the table
merged = df.merge(simbad_df, on="SIMBAD_name", how="left")

for col in ["ra", "dec", "U", "B", "V", "R", "I", "J", "H", "K", "sp_type"]:
    if col in merged.columns:
        merged[col] = merged[col].fillna("-")

# 5. Save
merged.to_csv("Nagata_1990_Lprime_with_Simbad.csv", index=False)
print("Wrote Nagata_1990_Lprime_with_Simbad.csv")


/tmp/ipykernel_935988/3181749844.py:18: DeprecationWarning: The notation 'flux(I)' is deprecated since 0.4.8 in favor of 'I'. You will see the column appearing with its new name in the output. See section on filters in https://astroquery.readthedocs.io/en/latest/simbad/simbad_evolution.html to see the new ways to interact with SIMBAD's fluxes.
  s.add_votable_fields(
/tmp/ipykernel_935988/3181749844.py:18: DeprecationWarning: The notation 'flux(V)' is deprecated since 0.4.8 in favor of 'V'. You will see the column appearing with its new name in the output. See section on filters in https://astroquery.readthedocs.io/en/latest/simbad/simbad_evolution.html to see the new ways to interact with SIMBAD's fluxes.
  s.add_votable_fields(
/tmp/ipykernel_935988/3181749844.py:18: DeprecationWarning: The notation 'flux(J)' is deprecated since 0.4.8 in favor of 'J'. You will see the column appearing with its new name in the output. See section on filters in https://astroquery.readthedocs.io/en/late

Wrote Nagata_1990_Lprime_with_Simbad.csv


In [5]:
!pip install astroquery


  Using cached html5lib-1.1-py2.py3-none-any.whl.metadata (16 kB)
  Using cached pyerfa-2.0.1.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.7 kB)
  Using cached jeepney-0.9.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached jaraco.classes-3.4.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached jaraco.context-6.0.1-py3-none-any.whl.metadata (4.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 99.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 101.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.3 MB/s  0:00:00
Using cached html5lib-1.1-py2.py3-none-any.whl (112 kB)
Using cached jeepney-0.9.0-py3-none-any.whl (49 kB)
Using cached pyerfa-2.0.1.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (738 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 86.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
import sys
print(sys.executable)

/home/rebeccaz/Github/ZetaPersei/.conda/bin/python
